# Malilangwe Wildlife Detector — WAID Fine-Tuning

Fine-tunes YOLOv11 on the WAID aerial wildlife dataset.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Edit the Settings cell below if needed
3. `Runtime → Run all`

**Hitting rate limits?** Use `EPOCHS=30`, `BATCH=8` (~50 min). Download `last.pt` at the end of each session and re-upload next time to resume.

## ⚙️ Settings — Edit These Before Running

In [1]:
# How many epochs to train this session
# First time: 30 (~50 min, safe for free Colab)
# Full run:   100 (~2-3 hrs, best results)
EPOCHS = 30

# Batch size — 8 is safe for free T4, 16 is faster but may crash
BATCH = 8

# Resuming from a previous session?
# False = start fresh | True = upload last.pt when prompted
RESUME = False

print(f'Settings: EPOCHS={EPOCHS}, BATCH={BATCH}, RESUME={RESUME}')

Settings: EPOCHS=30, BATCH=8, RESUME=False


## 1. Check GPU

In [2]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU, then re-run.')
print(r.stdout)

Mon Apr 13 14:33:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Repo & Install Dependencies

In [3]:
import os

REPO_URL = 'https://github.com/Tadiwa-M/wildlife-detector-malilangwe.git'
REPO_DIR = '/content/wildlife-detector-malilangwe'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Cloning into '/content/wildlife-detector-malilangwe'...
remote: Enumerating objects: 14466, done.
remote: Counting objects: 100% (14466/14466), done.
remote: Compressing objects: 100% (13982/13982), done.
remote: Total 14466 (delta 23), reused 14455 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (14466/14466), 4.13 MiB | 8.74 MiB/s, done.
Resolving deltas: 100% (23/23), done.
Working directory: /content/wildlife-detector-malilangwe


In [4]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.3 MB/s eta 0:00:00
Dependencies installed.


## 3. Upload Previous Checkpoint (Resume only)

**Skip if `RESUME = False`.** If resuming, upload your `last.pt` when prompted.

In [5]:
CHECKPOINT_PATH = None

if RESUME:
    from google.colab import files as colab_files
    from pathlib import Path
    print('Upload your last.pt checkpoint...')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded. Upload last.pt or set RESUME=False.')
    name = list(uploaded.keys())[0]
    dest = Path('runs/train/waid_yolo11n/weights')
    dest.mkdir(parents=True, exist_ok=True)
    os.rename(name, dest / 'last.pt')
    CHECKPOINT_PATH = str(dest / 'last.pt')
    print(f'Checkpoint ready: {CHECKPOINT_PATH}')
else:
    print('RESUME=False — starting fresh.')

RESUME=False — starting fresh.


## 4. Download WAID Images

In [6]:
import os
WAID_DIR = '/content/wildlife-detector-malilangwe/WAID'

if not os.path.exists(os.path.join(WAID_DIR, 'WAID', 'images')):
    print('Downloading WAID dataset (~2 GB)...')
    !git clone --depth=1 https://github.com/xiaohuicui/WAID.git {WAID_DIR}/WAID_repo
    !cp -r {WAID_DIR}/WAID_repo/WAID/images {WAID_DIR}/WAID/images
    print('Done.')
else:
    print('WAID images already present.')

Cloning into '/content/wildlife-detector-malilangwe/WAID/WAID_repo'...
remote: Enumerating objects: 28744, done.
remote: Counting objects: 100% (28744/28744), done.
remote: Compressing objects: 100% (28288/28288), done.
remote: Total 28744 (delta 4), reused 28742 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (28744/28744), 1.50 GiB | 19.39 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Updating files: 100% (28734/28734), done.
Done.


## 5. Validate Dataset & Generate YAML

In [7]:
import sys
sys.path.insert(0, REPO_DIR)

from src.config import load_config
from src.data.dataset import validate_dataset, get_class_distribution, generate_dataset_yaml
from src.utils.logging_setup import setup_logging

cfg = load_config()
setup_logging(cfg)

stats = validate_dataset(cfg)
print(f"\nTotal images : {stats['total_images']:,}")
print(f"Total labels : {stats['total_labels']:,}")

print('\nClass distribution (train):')
dist = get_class_distribution(cfg, split='train')
for name, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {name:<12s} {count:>8,}')

dataset_yaml = generate_dataset_yaml(cfg)
print('\nDataset YAML:', dataset_yaml)

2026-04-13 14:35:16,587 | INFO     | src.data.dataset |   train  — images: 10056  labels: 10056  orphan_img: 0  orphan_lbl: 0
2026-04-13 14:35:16,663 | INFO     | src.data.dataset |   val    — images: 2873  labels: 2873  orphan_img: 0  orphan_lbl: 0
2026-04-13 14:35:16,702 | INFO     | src.data.dataset |   test   — images: 1437  labels: 1437  orphan_img: 0  orphan_lbl: 0
2026-04-13 14:35:16,703 | INFO     | src.data.dataset | Dataset totals — images: 14366, labels: 14366

Total images : 14,366
Total labels : 14,366

Class distribution (train):
2026-04-13 14:35:20,678 | INFO     | src.data.dataset | Class distribution (train): {'sheep': 91496, 'cattle': 44245, 'seal': 15762, 'camelus': 4676, 'kiang': 3312, 'zebra': 3792}
  sheep          91,496
  cattle         44,245
  seal           15,762
  camelus         4,676
  zebra           3,792
  kiang           3,312
2026-04-13 14:35:20,681 | INFO     | src.data.dataset | Dataset YAML written to /content/wildlife-detector-malilangwe/data/wai

## 6. Train

Watch mAP50 climb each epoch. Early stopping kicks in after 15 epochs with no improvement.

In [8]:
from ultralytics import YOLO
from pathlib import Path
import shutil

train_cfg = cfg.training
det_cfg = cfg.detection

if RESUME:
    ckpt = Path(DRIVE_DIR) / 'waid_last.pt'
    if not ckpt.exists():
        raise FileNotFoundError(f'No checkpoint in Drive at {ckpt}. Set RESUME=False to start fresh.')
    print(f'Resuming from {ckpt}')
    model = YOLO(str(ckpt))
    resume_flag = True
else:
    print(f'Starting fresh from pretrained {det_cfg.model_variant}.pt')
    model = YOLO(f'{det_cfg.model_variant}.pt')
    resume_flag = False

results = model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=int(train_cfg.image_size),
    optimizer=str(train_cfg.optimizer),
    lr0=float(train_cfg.learning_rate),
    weight_decay=float(train_cfg.weight_decay),
    patience=int(train_cfg.patience),
    device=0,
    hsv_h=float(train_cfg.augmentation.hsv_h),
    hsv_s=float(train_cfg.augmentation.hsv_s),
    hsv_v=float(train_cfg.augmentation.hsv_v),
    flipud=float(train_cfg.augmentation.flipud),
    fliplr=float(train_cfg.augmentation.fliplr),
    mosaic=float(train_cfg.augmentation.mosaic),
    mixup=float(train_cfg.augmentation.mixup),
    scale=float(train_cfg.augmentation.scale),
    project='runs/train',
    name='waid_yolo11n',
    exist_ok=True,
    resume=resume_flag,
)

# Save BOTH best.pt and last.pt to Drive immediately after training
w = Path('runs/train/waid_yolo11n/weights')
for fname in ['best.pt', 'last.pt']:
    src = w / fname
    if src.exists():
        dest = Path(DRIVE_DIR) / f'waid_{fname}'
        shutil.copy(src, dest)
        print(f'Saved {fname} → Drive: {dest}')

print('\nTraining complete!')
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting fresh from pretrained yolo11n.pt
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/wildlife-detector-malilangwe/data/waid.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, f

## 7. Evaluate

In [9]:
from pathlib import Path
from ultralytics import YOLO

# Check local run first, fall back to Drive
best = Path('runs/train/waid_yolo11n/weights/best.pt')
if not best.exists():
    best = Path(DRIVE_DIR) / 'waid_best.pt'

if best.exists():
    print(f'Loading weights from: {best}')
    eval_model = YOLO(str(best))
    metrics = eval_model.val(
        data=str(dataset_yaml), split='test',
        conf=float(det_cfg.confidence_threshold),
        iou=float(det_cfg.iou_threshold),
        imgsz=int(det_cfg.image_size),
        device=0, plots=True,
    )
    print(f'mAP@50:    {metrics.box.map50:.4f}')
    print(f'mAP@50-95: {metrics.box.map:.4f}')
    print(f'Precision:  {metrics.box.mp:.4f}')
    print(f'Recall:     {metrics.box.mr:.4f}')
else:
    print('No weights found — did training finish?')

best.pt not found — did training finish?


## 8. Download Weights

- `waid_best.pt` — best checkpoint overall (use this locally when done)
- `waid_last.pt` — most recent epoch (re-upload this to resume next session)

In [12]:
from google.colab import files
import shutil
from pathlib import Path

w = Path('runs/train/waid_yolo11n/weights')
for fname, dest in [('best.pt', '/content/waid_best.pt'), ('last.pt', '/content/waid_last.pt')]:
    src = w / fname
    if src.exists():
        shutil.copy(src, dest)
        files.download(dest)
        print(f'Downloading {fname}...')

print('\nDone!')
print('  Resume next session: set RESUME=True, re-upload waid_last.pt')
print('  Finished training:   put waid_best.pt in your local weights/ folder')


Done!
  Resume next session: set RESUME=True, re-upload waid_last.pt
  Finished training:   put waid_best.pt in your local weights/ folder


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR    = '/content/drive/MyDrive/malilangwe_weights'
DATASETS_DIR = '/content/drive/MyDrive/datasets'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted.')
print(f'  Weights  : {DRIVE_DIR}')
print(f'  Datasets : {DATASETS_DIR}')

## 9. Test on a Custom Image (Optional)

In [11]:
from google.colab import files as colab_files
from IPython.display import Image, display
from pathlib import Path
from ultralytics import YOLO
import glob

# Check local run first, fall back to Drive
best = Path('runs/train/waid_yolo11n/weights/best.pt')
if not best.exists():
    best = Path(DRIVE_DIR) / 'waid_best.pt'

if not best.exists():
    print('No weights found. Run training first.')
else:
    print(f'Using weights: {best}')
    print('Upload an aerial image...')
    uploaded = colab_files.upload()
    if uploaded:
        img_path = list(uploaded.keys())[0]
        m = YOLO(str(best))
        res = m.predict(source=img_path, conf=0.15, save=True,
                        project='/content/test_output', name='result', exist_ok=True)
        for r in res:
            print(f'Detections: {len(r.boxes)}')
            for box in r.boxes:
                print(f'  {r.names[int(box.cls[0])]}: {float(box.conf[0]):.0%}')
        saved = glob.glob('/content/test_output/result/*.jpg') + glob.glob('/content/test_output/result/*.png')
        if saved:
            display(Image(saved[0]))

Upload an aerial image...


Saving elly.png to elly.png


FileNotFoundError: [Errno 2] No such file or directory: 'runs/train/waid_yolo11n/weights/best.pt'

---
## Phase A+ — Merged Dataset Training

Runs after WAID training is done. Merges WAID + AED + Liege + Savanna
into a unified 7-class dataset and fine-tunes from `waid_best.pt`.

**Run cells in order:** Extract → Inspect → Convert AED → Check Liege → Inspect Savanna → Merge → Train

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Phase A+  ONE-SHOT PIPELINE
# Run this single cell to go from a fresh Colab session all the way to a trained
# merged model. Each expensive step is skipped automatically if already done.
#
# Prerequisites — files in the ROOT of your Google Drive (MyDrive):
#   AED.tar.gz            Aerial Elephant Dataset
#   general_dataset.zip   Liege African Mammals
#   data.zip              Savanna Wildlife
#   malilangwe_weights/waid_best.pt   your trained WAID model
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, zipfile, tarfile
import pandas as pd
from collections import defaultdict
from pathlib import Path
from PIL import Image

# ── 1. SETUP ────────────────────────────────────────────────────────────────
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_DIR    = '/content/drive/MyDrive/malilangwe_weights'
DATASETS_DIR = '/content/drive/MyDrive'   # zip/tar files live here
REPO_URL     = 'https://github.com/Tadiwa-M/wildlife-detector-malilangwe.git'
REPO_DIR     = '/content/wildlife-detector-malilangwe'
REPO_BRANCH  = 'claude/code-review-assessment-WDnuF'
MERGED_OUT   = f'{DRIVE_DIR}/merged'

os.makedirs(DRIVE_DIR,  exist_ok=True)
os.makedirs(MERGED_OUT, exist_ok=True)

if not os.path.exists(REPO_DIR):
    print('Cloning repo ...')
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch origin {REPO_BRANCH} --quiet')
    os.system(f'git -C {REPO_DIR} checkout {REPO_BRANCH} --quiet')
    os.system(f'git -C {REPO_DIR} pull --quiet')
sys.path.insert(0, REPO_DIR)

os.system(f'pip install -q -r {REPO_DIR}/requirements.txt')
print(f'Repo ready at {REPO_DIR} (branch: {REPO_BRANCH})')

def _nfiles(path):
    return sum(len(fs) for _, _, fs in os.walk(path))

# ── 2. EXTRACT ──────────────────────────────────────────────────────────────
# Archives are copied from Drive to local disk before extraction so that the
# Drive FUSE mount doesn't drop during a 16 GB streaming read.
for name, drive_src, dst, fmt in [
    ('AED',     f'{DATASETS_DIR}/AED.tar.gz',          '/content/AED',     'tar'),
    ('Liege',   f'{DATASETS_DIR}/general_dataset.zip', '/content/liege',   'zip'),
    ('Savanna', f'{DATASETS_DIR}/data.zip',            '/content/savanna', 'zip'),
]:
    os.makedirs(dst, exist_ok=True)
    n = _nfiles(dst)
    if n >= 100:
        print(f'{name}: already extracted ({n:,} files)')
        continue
    gb = Path(drive_src).stat().st_size / 1e9
    local_archive = f'/content/{Path(drive_src).name}'
    if not os.path.exists(local_archive):
        print(f'Copying {name} ({gb:.1f} GB) from Drive to local disk ...')
        shutil.copy2(drive_src, local_archive)
        print(f'  Copy done')
    print(f'Extracting {name} ...')
    if fmt == 'tar':
        with tarfile.open(local_archive, 'r:gz') as tf:
            tf.extractall(dst, filter='data')
    else:
        with zipfile.ZipFile(local_archive, 'r') as zf:
            zf.extractall(dst)
    os.remove(local_archive)   # free local disk — extracted files stay
    print(f'  Done: {_nfiles(dst):,} files')

# ── 3. CONVERT LIEGE (COCO JSON → YOLO) ────────────────────────────────────
LIEGE_YOLO   = '/content/liege_yolo'
LIEGE_CAT_MAP = {
    1: 3,  # Alcelaphinae → antelope
    2: 2,  # Buffalo      → buffalo
    3: 3,  # Kob          → antelope
    4: 6,  # Warthog      → other
    5: 3,  # Waterbuck    → antelope
    6: 0,  # Elephant     → elephant
}

if _nfiles(LIEGE_YOLO) < 50:
    print('\nConverting Liege COCO → YOLO ...')
    liege_root = Path('/content/liege/general_dataset')
    json_dir   = liege_root / 'groundtruth' / 'json' / 'big_size'
    if not json_dir.exists():
        json_dir = liege_root / 'groundtruth' / 'json'

    total = 0
    for jf in sorted(json_dir.rglob('*.json')):
        with open(jf) as f:
            data = json.load(f)

        split = None
        for s in ['train', 'val', 'test']:
            sd = liege_root / s
            if not sd.exists():
                continue
            if {img['file_name'] for img in data['images']} & {f.name for f in sd.iterdir() if f.is_file()}:
                split = s
                break
        if split is None:
            for s in ['train', 'val', 'test']:
                if s in jf.stem.lower():
                    split = s
                    break
        if split is None:
            print(f'  Cannot determine split for {jf.name} — skip')
            continue

        out_img = Path(LIEGE_YOLO) / 'images' / split
        out_lbl = Path(LIEGE_YOLO) / 'labels' / split
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)

        ann_by_img = defaultdict(list)
        for ann in data['annotations']:
            if not ann.get('iscrowd', 0):
                ann_by_img[ann['image_id']].append(ann)

        copied = 0
        for img_info in data['images']:
            fname = img_info['file_name']
            iw, ih = img_info['width'], img_info['height']
            src_img = liege_root / split / fname
            if not src_img.exists():
                continue
            anns = ann_by_img.get(img_info['id'], [])
            if not anns:
                continue
            lines = []
            for ann in anns:
                cid = ann['category_id']
                if cid not in LIEGE_CAT_MAP:
                    continue
                cls = LIEGE_CAT_MAP[cid]
                x, y, bw, bh = ann['bbox']
                cx = max(0.0, min(1.0, (x + bw / 2) / iw))
                cy = max(0.0, min(1.0, (y + bh / 2) / ih))
                nw = max(0.0, min(1.0, bw / iw))
                nh = max(0.0, min(1.0, bh / ih))
                lines.append(f'{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
            if lines:
                with open(out_lbl / f'{Path(fname).stem}.txt', 'w') as f:
                    f.write('\n'.join(lines) + '\n')
                shutil.copy2(src_img, out_img / fname)
                copied += 1
        total += copied
        print(f'  {jf.name} → {split}: {copied} images')
    print(f'  Liege done: {total} images')
else:
    print(f'Liege YOLO already converted ({_nfiles(LIEGE_YOLO):,} files)')

LIEGE_PATH = LIEGE_YOLO

# ── 4. CONVERT SAVANNA (CSV → YOLO) ────────────────────────────────────────
SAVANNA_YOLO = '/content/savanna_yolo'
SPECIES_MAP  = {'Elephant': 0, 'Zebra': 1, 'Giraffe': 5}

if _nfiles(SAVANNA_YOLO) < 50:
    print('\nConverting Savanna CSV → YOLO ...')
    savanna_root = Path('/content/savanna')

    def _savanna_split(csv_path, split, img_dir):
        df = pd.read_csv(csv_path, names=['file', 'x1', 'y1', 'x2', 'y2', 'species'])
        out_img = Path(SAVANNA_YOLO) / 'images' / split
        out_lbl = Path(SAVANNA_YOLO) / 'labels' / split
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)
        copied = skipped = 0
        for fname, grp in df.groupby('file'):
            img_path = savanna_root / fname
            if not img_path.exists():
                img_path = img_dir / Path(fname).name
            if not img_path.exists():
                skipped += 1
                continue
            with Image.open(img_path) as img:
                iw, ih = img.size
            lines = []
            for _, row in grp.iterrows():
                cls = SPECIES_MAP.get(row['species'])
                if cls is None:
                    continue
                cx = max(0.0, min(1.0, (row['x1'] + row['x2']) / 2 / iw))
                cy = max(0.0, min(1.0, (row['y1'] + row['y2']) / 2 / ih))
                nw = max(0.0, min(1.0, (row['x2'] - row['x1']) / iw))
                nh = max(0.0, min(1.0, (row['y2'] - row['y1']) / ih))
                lines.append(f'{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
            if lines:
                with open(out_lbl / f'{Path(img_path.name).stem}.txt', 'w') as f:
                    f.write('\n'.join(lines) + '\n')
                shutil.copy2(img_path, out_img / img_path.name)
                copied += 1
        print(f'  {split}: {copied} converted, {skipped} skipped')

    _savanna_split(savanna_root / 'annotations_val.csv',  'val',  savanna_root / 'val')
    _savanna_split(savanna_root / 'annotations_test.csv', 'test', savanna_root / 'test')

    # Train uses raw annotations_images.csv filtered to images in train/
    train_imgs = {f.name for f in (savanna_root / 'train').iterdir() if f.is_file()}
    df_raw     = pd.read_csv(savanna_root / 'annotations_images.csv')
    df_train   = df_raw[df_raw['FILE'].isin(train_imgs)].rename(
                     columns={'FILE': 'file', 'SPECIES': 'species'})
    out_img = Path(SAVANNA_YOLO) / 'images' / 'train'
    out_lbl = Path(SAVANNA_YOLO) / 'labels' / 'train'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    copied = skipped = 0
    for fname, grp in df_train.groupby('file'):
        img_path = savanna_root / 'train' / fname
        if not img_path.exists():
            skipped += 1
            continue
        with Image.open(img_path) as img:
            iw, ih = img.size
        lines = []
        for _, row in grp.iterrows():
            cls = SPECIES_MAP.get(row['species'])
            if cls is None:
                continue
            cx = max(0.0, min(1.0, (row['x1'] + row['x2']) / 2 / iw))
            cy = max(0.0, min(1.0, (row['y1'] + row['y2']) / 2 / ih))
            nw = max(0.0, min(1.0, (row['x2'] - row['x1']) / iw))
            nh = max(0.0, min(1.0, (row['y2'] - row['y1']) / ih))
            lines.append(f'{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        if lines:
            with open(out_lbl / f'{Path(fname).stem}.txt', 'w') as f:
                f.write('\n'.join(lines) + '\n')
            shutil.copy2(img_path, out_img / fname)
            copied += 1
    print(f'  train: {copied} converted, {skipped} skipped')
    print('  Savanna done')
else:
    print(f'Savanna YOLO already converted ({_nfiles(SAVANNA_YOLO):,} files)')

SAVANNA_PATH = SAVANNA_YOLO

# ── 5. MERGE (local disk → then copy to Drive) ──────────────────────────────
# Write to local disk first — Drive FUSE is unreliable for thousands of small
# file writes. Copy the finished directory to Drive in one rsync pass.
LOCAL_MERGED = '/content/merged'
merge_train  = Path(LOCAL_MERGED) / 'images' / 'train'
n_merged     = len(list(merge_train.iterdir())) if merge_train.exists() else 0

if n_merged < 10:
    print('\nMerging datasets (local disk) ...')
    os.makedirs(LOCAL_MERGED, exist_ok=True)
    import subprocess
    cmd = [
        'python', f'{REPO_DIR}/scripts/merge_datasets.py',
        '--waid',    f'{REPO_DIR}/WAID/WAID',
        '--liege',   LIEGE_PATH,
        '--savanna', SAVANNA_PATH,
        '--output',  LOCAL_MERGED,
    ]
    print('CMD:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Merge script failed (exit {result.returncode}). See output above.')
    n_merged = len(list(merge_train.iterdir())) if merge_train.exists() else 0
    print(f'Merge done: {n_merged:,} train images')

    # Copy merged directory to Drive
    print('Copying merged dataset to Drive ...')
    ret = os.system(f'rsync -a --info=progress2 {LOCAL_MERGED}/ {MERGED_OUT}/')
    if ret != 0:
        print(f'rsync failed (exit {ret}), trying shutil.copytree ...')
        shutil.copytree(LOCAL_MERGED, MERGED_OUT, dirs_exist_ok=True)
    print(f'Drive copy done: {MERGED_OUT}')
else:
    print(f'\nMerge already done ({n_merged:,} train images on local disk)')
    # Make sure Drive copy is also up to date
    drive_train = Path(MERGED_OUT) / 'images' / 'train'
    n_drive = len(list(drive_train.iterdir())) if drive_train.exists() else 0
    if n_drive < n_merged:
        print('Syncing to Drive ...')
        os.system(f'rsync -a {LOCAL_MERGED}/ {MERGED_OUT}/')

# merged.yaml — path points at Drive so it persists across sessions
import yaml
merged_yaml = Path(REPO_DIR) / 'data' / 'merged.yaml'
merged_yaml.parent.mkdir(parents=True, exist_ok=True)
with open(merged_yaml, 'w') as f:
    yaml.dump({
        'path':  MERGED_OUT,
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc':    7,
        'names': ['elephant', 'zebra', 'buffalo', 'antelope', 'cattle', 'giraffe', 'other'],
    }, f, default_flow_style=False, sort_keys=False)
n_drive_final = len(list((Path(MERGED_OUT) / 'images' / 'train').iterdir())) if (Path(MERGED_OUT) / 'images' / 'train').exists() else 0
print(f'merged.yaml written  ({n_drive_final:,} train images in Drive)')
if n_drive_final == 0:
    raise RuntimeError('Merge produced 0 images — check errors above before training.')

# ── 6. TRANSFER LEARN ───────────────────────────────────────────────────────
from ultralytics import YOLO

base_weights = Path(DRIVE_DIR) / 'waid_best.pt'
if not base_weights.exists():
    print(f'\nSkipping training — {base_weights} not found.')
    print('Upload waid_best.pt to Drive/malilangwe_weights/ and re-run.')
else:
    print(f'\nTransfer learning  {base_weights} → merged ({n_merged:,} train imgs)')
    model   = YOLO(str(base_weights))
    results = model.train(
        data=str(merged_yaml),
        epochs=30,
        batch=8,
        imgsz=640,
        optimizer='AdamW',
        lr0=0.0005,
        freeze=10,
        patience=10,
        device=0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        project='runs/train',
        name='merged_transfer',
        exist_ok=True,
    )
    w = Path('runs/train/merged_transfer/weights')
    for fname in ['best.pt', 'last.pt']:
        src = w / fname
        if src.exists():
            dest = Path(DRIVE_DIR) / f'merged_{fname}'
            shutil.copy(src, dest)
            print(f'Saved → Drive: {dest}')
    print('Done!  Best mAP50:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))


### PA+·1 — Extract Datasets

In [ ]:
import os
from google.colab import drive

# Mount Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_DIR    = '/content/drive/MyDrive/malilangwe_weights'
DATASETS_DIR = '/content/drive/MyDrive/datasets'
os.makedirs(DRIVE_DIR, exist_ok=True)

os.makedirs('/content/AED',     exist_ok=True)
os.makedirs('/content/liege',   exist_ok=True)
os.makedirs('/content/savanna', exist_ok=True)

print('Extracting AED.tar.gz ...')
os.system(f'tar -xzf "{DATASETS_DIR}/AED.tar.gz" -C /content/AED')
print('Extracting general_dataset.zip (Liege) ...')
os.system(f'unzip -q "{DATASETS_DIR}/general_dataset.zip" -d /content/liege')
print('Extracting data.zip (Savanna) ...')
os.system(f'unzip -q "{DATASETS_DIR}/data.zip" -d /content/savanna')
print('Done.')

### PA+·2 — Inspect Structures

In [ ]:
from pathlib import Path
from collections import Counter

for name, path in [('AED', '/content/AED'), ('Liege', '/content/liege'), ('Savanna', '/content/savanna')]:
    p = Path(path)
    print(f'\n{"="*55}')
    print(f'{name}  ({path})')
    print('='*55)
    all_files = list(p.rglob('*'))
    dirs = sorted(d for d in all_files if d.is_dir())
    for d in dirs[:20]:
        print(' ', str(d).replace(path, ''))
    exts = Counter(f.suffix for f in all_files if f.is_file())
    print('  File types:', dict(exts.most_common(8)))

### PA+·3 — Convert AED (VOC XML → YOLO)

In [ ]:
import shutil, random
from pathlib import Path

aed_root = Path('/content/AED')

# Handle single subfolder after extraction
subdirs = [d for d in aed_root.iterdir() if d.is_dir()]
if len(subdirs) == 1 and not (aed_root / 'images').exists():
    aed_root = subdirs[0]

xml_files  = list(aed_root.rglob('*.xml'))
lbl_dirs   = list(aed_root.rglob('labels'))

if lbl_dirs:
    print('AED already in YOLO format — no conversion needed.')
    AED_PATH = str(lbl_dirs[0].parent)
elif xml_files:
    print(f'Converting {len(xml_files)} VOC XML files to YOLO ...')
    !pip install -q pylabel
    from pylabel import importer
    annot_dir = str(xml_files[0].parent)
    out_dir   = str(aed_root / 'yolo')
    ds = importer.ImportVOC(annot_dir)
    ds.export.ExportToYoloV5(output_path=out_dir)
    aed_root = Path(out_dir)
    # 80/10/10 split
    imgs = [f for f in (aed_root / 'images').iterdir() if f.is_file()]
    random.seed(42); random.shuffle(imgs)
    n = len(imgs)
    splits = {'train': imgs[:int(n*.8)], 'val': imgs[int(n*.8):int(n*.9)], 'test': imgs[int(n*.9):]}
    for split, files in splits.items():
        (aed_root / 'images' / split).mkdir(parents=True, exist_ok=True)
        (aed_root / 'labels' / split).mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.move(str(f), aed_root / 'images' / split / f.name)
            lbl = aed_root / 'labels' / (f.stem + '.txt')
            if lbl.exists():
                shutil.move(str(lbl), aed_root / 'labels' / split / lbl.name)
    print(f'Split: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')
    AED_PATH = str(aed_root)
else:
    print('ERROR: No XML or labels/ found — check inspect output above.')
    AED_PATH = None

print(f'AED_PATH = {AED_PATH}')

### PA+·4 — Verify Liege Classes

In [ ]:
from pathlib import Path

liege_root = Path('/content/liege')

# Handle single subfolder after extraction
if not (liege_root / 'classes.txt').exists():
    subdirs = [d for d in liege_root.iterdir() if d.is_dir()]
    if subdirs:
        liege_root = subdirs[0]

classes_file = liege_root / 'classes.txt'
if classes_file.exists():
    print('Liege classes.txt:')
    with open(classes_file) as f:
        for i, line in enumerate(f):
            print(f'  {i}: {line.strip()}')
    print()
    print('Expected order (config/merged_classes.yaml):')
    print('  0: zebra | 1: elephant | 2: buffalo | 3: kob | 4: topi | 5: warthog | 6: waterbuck')
    print()
    print('If the order differs, update dataset_mappings.liege in config/merged_classes.yaml.')
else:
    print(f'No classes.txt found in {liege_root}. Check inspect output.')

LIEGE_PATH = str(liege_root)
print(f'LIEGE_PATH = {LIEGE_PATH}')

### PA+·5 — Inspect Savanna (share output to get converter written)

In [ ]:
from pathlib import Path

savanna_root = Path('/content/savanna')

# Handle single subfolder
subdirs = [d for d in savanna_root.iterdir() if d.is_dir()]
if len(subdirs) == 1:
    savanna_root = subdirs[0]

print('=== Directory structure ===')
all_items = sorted(savanna_root.rglob('*'))
for item in all_items[:60]:
    print(str(item).replace(str(savanna_root), ''))

print('\n=== Python scripts ===')
for py in sorted(savanna_root.rglob('*.py')):
    print(f'\n--- {py.name} ---')
    with open(py) as f:
        print(f.read()[:800])

print('\n=== README ===')
for readme in sorted(savanna_root.rglob('README*')):
    with open(readme) as f:
        print(f.read()[:800])

# Placeholder — replace once converter is written
SAVANNA_PATH = None
print(f'\nSAVANNA_PATH = {SAVANNA_PATH}  (set after converter is written)')

In [ ]:
# Paths are set by the cells above — run them first.
# Output goes to Drive so merged data survives session disconnects
# and avoids filling Colab temp disk (currently ~92 GB used).
import os, sys
sys.path.insert(0, REPO_DIR)

MERGED_OUTPUT = '/content/drive/MyDrive/malilangwe_weights/merged'
os.makedirs(MERGED_OUTPUT, exist_ok=True)

cmd = (
    f'python {REPO_DIR}/scripts/merge_datasets.py'
    f' --waid {REPO_DIR}/WAID/WAID'
    f' --liege {LIEGE_PATH}'
    f' --output {MERGED_OUTPUT}'
)
if SAVANNA_PATH and os.path.exists(SAVANNA_PATH):
    cmd += f' --savanna {SAVANNA_PATH}'

print('Running:', cmd)
os.system(cmd + ' 2>&1')
print('
Merge complete. Output at:', MERGED_OUTPUT)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil, os

MERGED_OUTPUT = '/content/drive/MyDrive/malilangwe_weights/merged'

# merged.yaml is written next to the merged dir by the merge script
merged_yaml = Path(REPO_DIR) / 'data' / 'merged.yaml'
if not merged_yaml.exists():
    # Fallback: generate yaml pointing at Drive output
    import yaml
    ds = {
        'path': MERGED_OUTPUT,
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc': 7,
        'names': ['elephant', 'zebra', 'buffalo', 'antelope', 'cattle', 'giraffe', 'other'],
    }
    merged_yaml.parent.mkdir(parents=True, exist_ok=True)
    with open(merged_yaml, 'w') as f:
        yaml.dump(ds, f, default_flow_style=False, sort_keys=False)
    print('Generated merged.yaml pointing at Drive output')

# Update path inside yaml to point at Drive (merge script uses repo-relative path)
import yaml
with open(merged_yaml) as f:
    ds = yaml.safe_load(f)
if not os.path.exists(ds.get('path', '')):
    ds['path'] = MERGED_OUTPUT
    with open(merged_yaml, 'w') as f:
        yaml.dump(ds, f, default_flow_style=False, sort_keys=False)
    print('Updated merged.yaml path → Drive')

# Load WAID-trained weights as starting point (transfer learning)
base_weights = Path(DRIVE_DIR) / 'waid_best.pt'
if not base_weights.exists():
    raise FileNotFoundError(f'WAID best.pt not found in Drive at {base_weights}. Upload it first.')

print(f'Transfer learning from: {base_weights}')
print(f'Dataset: {merged_yaml}')

model = YOLO(str(base_weights))
results = model.train(
    data=str(merged_yaml),
    epochs=30,
    batch=8,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.0005,       # lower LR for transfer learning
    freeze=10,        # freeze first 10 backbone layers
    patience=10,
    device=0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    project='runs/train',
    name='merged_transfer',
    exist_ok=True,
)

# Save to Drive immediately
w = Path('runs/train/merged_transfer/weights')
for fname in ['best.pt', 'last.pt']:
    src = w / fname
    if src.exists():
        dest = Path(DRIVE_DIR) / f'merged_{fname}'
        shutil.copy(src, dest)
        print(f'Saved {fname} → Drive: {dest}')

print('Transfer learning complete!')
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

drive_weights = Path(DRIVE_DIR)
for fname in ['merged_best.pt', 'merged_last.pt']:
    p = drive_weights / fname
    if p.exists():
        files.download(str(p))
        print(f'Downloading {fname}...')
    else:
        print(f'{fname} not in Drive yet')

print('\nPut merged_best.pt in your local weights/ folder and update config:')
print('  paths.default_model: weights/merged_best.pt')